# WanderBricks Advanced Business Intelligence

**Dataset:** `samples.wanderbricks (all tables)`

**Difficulty:** Hard

**Topics:** advanced window functions, complex joins, sessionization, statistical analysis

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

bookings   = spark.table("samples.wanderbricks.bookings")
users      = spark.table("samples.wanderbricks.users")
properties = spark.table("samples.wanderbricks.properties")
reviews    = spark.table("samples.wanderbricks.reviews")
payments   = spark.table("samples.wanderbricks.payments")
hosts      = spark.table("samples.wanderbricks.hosts")
clickstream = spark.table("samples.wanderbricks.clickstream")

## Problem 1

**Dynamic Pricing Analysis.**

Compare the actual charged price per night with the property's `base_price`. For each confirmed booking, compute `charged_per_night = total_amount / DATEDIFF(check_out, check_in)`. Then aggregate per property: average charged per night, `price_premium_pct = (avg_charged_per_night - base_price) / base_price * 100`, and booking count.

Business context: Properties consistently charging above their listed base price indicate hidden fees or manual price overrides — the trust team uses this analysis to identify hosts who may be violating transparent-pricing policies.

**Expected output columns:** `property_id`, `title`, `property_type`, `base_price`, `avg_charged_per_night`, `price_premium_pct`, `booking_count`

In [ ]:
# Problem 1 — write your solution here
# Assign your result to: result_1
#   confirmed = bookings.filter(F.col('status') == 'confirmed')
#   nights = F.datediff(F.col('check_out'), F.col('check_in'))
#   with_cpn = confirmed.withColumn('charged_per_night',
#       F.when(nights > 0, F.col('total_amount') / nights))
#   agg = with_cpn.groupBy('property_id').agg(
#       F.avg('charged_per_night').alias('avg_charged_per_night'),
#       F.count('booking_id').alias('booking_count')
#   )
#   result_1 = agg.join(properties, 'property_id')\
#              .withColumn('price_premium_pct',
#                  (F.col('avg_charged_per_night') - F.col('base_price')) / F.col('base_price') * 100)\
#              .select('property_id', 'title', 'property_type', 'base_price',
#                      'avg_charged_per_night', 'price_premium_pct', 'booking_count')

result_1 = None  # replace this

In [ ]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None — did you assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
for col in ['property_id', 'title', 'property_type', 'base_price',
            'avg_charged_per_night', 'price_premium_pct', 'booking_count']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 7, f"Expected exactly 7 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert result_1.agg(F.min('booking_count')).collect()[0][0] > 0, "booking_count must be positive"
assert result_1.agg(F.min('base_price')).collect()[0][0] > 0, "base_price must be positive"
null_prem = result_1.filter(F.col('price_premium_pct').isNull()).count()
assert null_prem == 0, f"Found {null_prem} rows with null price_premium_pct"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

**Review Score Trajectory.**

For each property, compare the average rating of the first 5 reviews (by `created_at`) vs the last 5 reviews. Compute `early_avg_rating`, `recent_avg_rating`, and a `rating_trend` label: `'improving'` if recent > early by > 0.2, `'declining'` if recent < early by > 0.2, otherwise `'stable'`. Include total review count.

Business context: Review trajectory analysis helps the quality team identify properties in decline — those with falling ratings receive a proactive outreach from the host success team before they lose search ranking.


**Expected output columns:** `property_id`, `early_avg_rating`, `recent_avg_rating`, `rating_trend`, `total_reviews`

In [ ]:
# Problem 2 — write your solution here
# Assign your result to: result_2
#   non_deleted = reviews.filter(F.col('is_deleted') == False)
#   w_asc = Window.partitionBy('property_id').orderBy('created_at')
#   w_desc = Window.partitionBy('property_id').orderBy(F.desc('created_at'))
#   ranked = non_deleted.withColumn('early_rank', F.rank().over(w_asc))\
#                       .withColumn('recent_rank', F.rank().over(w_desc))
#   early_avg = ranked.filter(F.col('early_rank') <= 5).groupBy('property_id')\
#               .agg(F.avg('rating').alias('early_avg_rating'))
#   recent_avg = ranked.filter(F.col('recent_rank') <= 5).groupBy('property_id')\
#                .agg(F.avg('rating').alias('recent_avg_rating'))
#   total = non_deleted.groupBy('property_id').agg(F.count('review_id').alias('total_reviews'))
#   result_2 = early_avg.join(recent_avg, 'property_id').join(total, 'property_id')\
#              .withColumn('rating_trend',
#                  F.when(F.col('recent_avg_rating') - F.col('early_avg_rating') > 0.2, 'improving')\
#                   .when(F.col('early_avg_rating') - F.col('recent_avg_rating') > 0.2, 'declining')\
#                   .otherwise('stable'))

result_2 = None  # replace this

In [ ]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None — did you assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
for col in ['property_id', 'early_avg_rating', 'recent_avg_rating', 'rating_trend', 'total_reviews']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
valid_trends = {'improving', 'declining', 'stable'}
invalid_trends = result_2.filter(~F.col('rating_trend').isin(list(valid_trends))).count()
assert invalid_trends == 0, f"Found {invalid_trends} rows with invalid rating_trend values"
assert result_2.agg(F.min('total_reviews')).collect()[0][0] > 0, "total_reviews must be positive"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

**Superhost Identification.**

Identify hosts who qualify as 'superhosts' based on ALL of the following criteria:
1. `is_verified = true`
2. Average property rating (from non-deleted reviews) >= 4.5
3. At least 10 confirmed bookings
4. Cancellation rate < 5%

Join `hosts` → `properties` → `bookings` → `reviews`. Compute each metric and flag qualifying hosts with `is_superhost = true`.

Business context: The Superhost programme drives premium listing visibility and is WanderBricks' primary quality-assurance lever. Accurate identification ensures the badge is credible and maintains high consumer trust.

**Expected output columns:** `host_id`, `name`, `avg_rating`, `confirmed_bookings`, `cancellation_rate_pct`, `is_superhost`

In [ ]:
# Problem 3 — write your solution here
# Assign your result to: result_3
#   1. Join properties -> bookings on property_id
#   2. Per host_id: confirmed_bookings = sum(status='confirmed')
#                   cancellation_rate = sum(status='cancelled') / count(*) * 100
#   3. Per host_id: avg_rating from reviews (non-deleted)
#      via properties -> reviews join
#   4. Join with hosts for is_verified + name
#   5. is_superhost = is_verified AND avg_rating >= 4.5 AND confirmed_bookings >= 10
#                     AND cancellation_rate < 5

result_3 = None  # replace this

In [ ]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None — did you assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
for col in ['host_id', 'name', 'avg_rating', 'confirmed_bookings',
            'cancellation_rate_pct', 'is_superhost']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 6, f"Expected exactly 6 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
# All superhosts must satisfy all 4 criteria
superhost_df = result_3.filter(F.col('is_superhost') == True)
if superhost_df.count() > 0:
    bad_rating = superhost_df.filter(F.col('avg_rating') < 4.5).count()
    assert bad_rating == 0, f"Found {bad_rating} superhosts with avg_rating < 4.5"
    bad_bookings = superhost_df.filter(F.col('confirmed_bookings') < 10).count()
    assert bad_bookings == 0, f"Found {bad_bookings} superhosts with < 10 confirmed bookings"
assert result_3.agg(F.min('cancellation_rate_pct')).collect()[0][0] >= 0, "cancellation_rate_pct must be non-negative"
print(f"Problem 3 passed ✓  ({cnt} hosts evaluated)")

## Problem 4

**Cross-Property User Behaviour.**

Find users who have booked more than one distinct `property_type`. For each such user compute: the array of distinct property types booked, the count of distinct property types, and total spend (sum of `total_amount`). Join with `users` for the name. Include bookings of any status.

Business context: Multi-type bookers are the most valuable segment — they demonstrate higher travel frequency and lower acquisition costs per booking. The product team uses this data to build personalised cross-type recommendation features.


**Expected output columns:** `user_id`, `name`, `property_types_booked`, `type_count`, `total_spend`

In [ ]:
# Problem 4 — write your solution here
# Assign your result to: result_4
#   joined = bookings.join(properties.select('property_id', 'property_type'), 'property_id')
#   agg = joined.groupBy('user_id').agg(
#       F.collect_set('property_type').alias('property_types_booked'),
#       F.sum('total_amount').alias('total_spend')
#   ).withColumn('type_count', F.size('property_types_booked'))\
#    .filter(F.col('type_count') > 1)
#   result_4 = agg.join(users.select('user_id', 'name'), 'user_id')\
#              .select('user_id', 'name', 'property_types_booked', 'type_count', 'total_spend')

result_4 = None  # replace this

In [ ]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None — did you assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
for col in ['user_id', 'name', 'property_types_booked', 'type_count', 'total_spend']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
# All users should have type_count > 1
single_type = result_4.filter(F.col('type_count') <= 1).count()
assert single_type == 0, f"Found {single_type} users with type_count <= 1 — should be filtered out"
# property_types_booked should be an array
arr_type = result_4.schema['property_types_booked'].dataType
assert isinstance(arr_type, T.ArrayType), f"property_types_booked should be ArrayType, got {arr_type}"
assert result_4.agg(F.min('total_spend')).collect()[0][0] > 0, "total_spend must be positive"
print(f"Problem 4 passed ✓  ({cnt} multi-type users)")

## Problem 5

**Seasonal Demand Analysis.**

Compute booking count and total revenue per month of year (1–12) per `property_type` (using `check_in` date month). Use a window function to rank months within each property type by booking count descending (`demand_rank`).

Business context: Seasonal demand patterns allow the supply team to run targeted host recruitment campaigns before peak months — ensuring adequate inventory is available when demand is highest.

**Expected output columns:** `property_type`, `month`, `booking_count`, `total_revenue`, `demand_rank`

In [ ]:
# Problem 5 — write your solution here
# Assign your result to: result_5
#   joined = bookings.join(properties.select('property_id', 'property_type'), 'property_id')
#   with_month = joined.withColumn('month', F.month('check_in'))
#   agg = with_month.groupBy('property_type', 'month').agg(
#       F.count('booking_id').alias('booking_count'),
#       F.sum('total_amount').alias('total_revenue')
#   )
#   w = Window.partitionBy('property_type').orderBy(F.desc('booking_count'))
#   result_5 = agg.withColumn('demand_rank', F.rank().over(w))

result_5 = None  # replace this

In [ ]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None — did you assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
for col in ['property_type', 'month', 'booking_count', 'total_revenue', 'demand_rank']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
month_range = result_5.agg(F.min('month'), F.max('month')).collect()[0]
assert month_range[0] >= 1 and month_range[1] <= 12, "month must be in range 1-12"
assert result_5.agg(F.min('demand_rank')).collect()[0][0] == 1, "Minimum demand_rank should be 1"
assert result_5.agg(F.min('booking_count')).collect()[0][0] > 0, "booking_count must be positive"
print(f"Problem 5 passed ✓  ({cnt} rows)")

## Problem 6

**Payment vs Booking Amount Reconciliation.**

For each booking, compare the booking's `total_amount` with the sum of all completed payment amounts (`payments` where `status = 'completed'`). Compute `variance = total_paid - booking_amount` and `variance_pct = abs(variance) / booking_amount * 100`. Flag bookings as `is_reconciled = false` where `abs(variance_pct) > 1`.

Business context: Payment reconciliation is a critical finance control — unreconciled bookings represent revenue leakage or customer overcharging risk, both of which carry regulatory and reputational consequences.

**Expected output columns:** `booking_id`, `booking_amount`, `total_paid`, `variance`, `variance_pct`, `is_reconciled`

In [ ]:
# Problem 6 — write your solution here
# Assign your result to: result_6
#   completed_payments = payments.filter(F.col('status') == 'completed')
#   payment_totals = completed_payments.groupBy('booking_id').agg(
#       F.sum('amount').alias('total_paid')
#   )
#   result_6 = bookings.join(payment_totals, 'booking_id', 'left')\
#              .withColumn('total_paid', F.coalesce('total_paid', F.lit(0.0)))\
#              .withColumn('booking_amount', F.col('total_amount').cast('double'))\
#              .withColumn('variance', F.col('total_paid') - F.col('booking_amount'))\
#              .withColumn('variance_pct',
#                  F.when(F.col('booking_amount') != 0,
#                      F.abs(F.col('variance')) / F.col('booking_amount') * 100).otherwise(F.lit(None)))\
#              .withColumn('is_reconciled',
#                  F.when(F.col('variance_pct') <= 1, True).otherwise(False))\
#              .select('booking_id', 'booking_amount', 'total_paid',
#                      'variance', 'variance_pct', 'is_reconciled')

result_6 = None  # replace this

In [ ]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None — did you assign your DataFrame?"
assert hasattr(result_6, 'columns'), "result_6 must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
for col in ['booking_id', 'booking_amount', 'total_paid', 'variance', 'variance_pct', 'is_reconciled']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 6, f"Expected exactly 6 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
# is_reconciled should be boolean
bool_type = result_6.schema['is_reconciled'].dataType
assert isinstance(bool_type, T.BooleanType), f"is_reconciled should be BooleanType, got {bool_type}"
recon_count = result_6.filter(F.col('is_reconciled') == True).count()
unrecon_count = result_6.filter(F.col('is_reconciled') == False).count()
assert recon_count + unrecon_count <= cnt, "reconciled + unreconciled should not exceed total"
print(f"Problem 6 passed ✓  ({cnt} rows, {recon_count} reconciled, {unrecon_count} not reconciled)")

## Problem 7

**Property Recommendation Features.**

Build a feature table for a recommendation model. For each property compute:
- `avg_rating` — average review rating (non-deleted reviews)
- `review_count` — total review count
- `booking_count` — total confirmed bookings
- `occupancy_rate` — total booked nights / 365
- `avg_booking_value` — average booking total_amount
- `avg_stay_nights` — average DATEDIFF(check_out, check_in)

Join `properties` → `bookings` (confirmed) → `reviews` (non-deleted).

Business context: Recommendation systems require rich per-property features that encode both popularity (booking count, occupancy) and quality (rating, review count) signals to surface the most relevant properties for each user.

**Expected output columns:** `property_id`, `title`, `property_type`, `avg_rating`, `review_count`, `booking_count`, `occupancy_rate`, `avg_booking_value`, `avg_stay_nights`

In [ ]:
# Problem 7 — write your solution here
# Assign your result to: result_7
#   confirmed = bookings.filter(F.col('status') == 'confirmed')
#   non_deleted_reviews = reviews.filter(F.col('is_deleted') == False)
#   nights = F.datediff(F.col('check_out'), F.col('check_in'))
#   booking_agg = confirmed.groupBy('property_id').agg(
#       F.count('booking_id').alias('booking_count'),
#       F.sum(nights).alias('total_nights'),
#       F.avg('total_amount').alias('avg_booking_value'),
#       F.avg(nights).alias('avg_stay_nights')
#   )
#   review_agg = non_deleted_reviews.groupBy('property_id').agg(
#       F.avg('rating').alias('avg_rating'),
#       F.count('review_id').alias('review_count')
#   )
#   result_7 = properties.select('property_id', 'title', 'property_type')\
#              .join(booking_agg, 'property_id', 'left')\
#              .join(review_agg, 'property_id', 'left')\
#              .withColumn('occupancy_rate',
#                  F.coalesce(F.col('total_nights'), F.lit(0)) / 365)\
#              .select('property_id', 'title', 'property_type', 'avg_rating', 'review_count',
#                      'booking_count', 'occupancy_rate', 'avg_booking_value', 'avg_stay_nights')

result_7 = None  # replace this

In [ ]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None — did you assign your DataFrame?"
assert hasattr(result_7, 'columns'), "result_7 must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
for col in ['property_id', 'title', 'property_type', 'avg_rating', 'review_count',
            'booking_count', 'occupancy_rate', 'avg_booking_value', 'avg_stay_nights']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 9, f"Expected exactly 9 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
occ_max = result_7.agg(F.max('occupancy_rate')).collect()[0][0]
assert occ_max is None or occ_max >= 0, "occupancy_rate must be non-negative"
avg_r_max = result_7.agg(F.max('avg_rating')).collect()[0][0]
assert avg_r_max is None or avg_r_max <= 5.0, f"avg_rating must be <= 5.0, got {avg_r_max}"
print(f"Problem 7 passed ✓  ({cnt} properties)")

## Problem 8

**Host Revenue Concentration (Gini-like).**

Compute the total revenue earned per host (from confirmed bookings). Sort hosts by revenue descending, assign a `revenue_rank`, compute `host_percentile` (rank / total_hosts * 100), and `cumulative_revenue_pct` (running sum of each host's share of total revenue, ordered by rank ascending).

Business context: Revenue concentration analysis quantifies platform dependency on a small number of hosts — if the top 10% of hosts generate >50% of revenue, the platform faces significant supply-side concentration risk that must be mitigated through host diversification.


**Expected output columns:** `host_id`, `total_revenue`, `revenue_rank`, `cumulative_revenue_pct`, `host_percentile`

In [ ]:
# Problem 8 — write your solution here
# Assign your result to: result_8
#   confirmed = bookings.filter(F.col('status') == 'confirmed')
#   host_rev = confirmed.join(properties.select('property_id', 'host_id'), 'property_id')\
#              .groupBy('host_id').agg(F.sum('total_amount').alias('total_revenue'))
#   total_rev = host_rev.agg(F.sum('total_revenue')).collect()[0][0]
#   total_hosts = host_rev.count()
#   rank_w = Window.orderBy(F.desc('total_revenue'))
#   cum_w = Window.orderBy(F.desc('total_revenue')).rowsBetween(Window.unboundedPreceding, Window.currentRow)
#   result_8 = host_rev\
#       .withColumn('revenue_rank', F.rank().over(rank_w))\
#       .withColumn('revenue_pct', F.col('total_revenue') / total_rev * 100)\
#       .withColumn('cumulative_revenue_pct', F.sum('revenue_pct').over(cum_w))\
#       .withColumn('host_percentile', F.col('revenue_rank') / total_hosts * 100)\
#       .select('host_id', 'total_revenue', 'revenue_rank', 'cumulative_revenue_pct', 'host_percentile')

result_8 = None  # replace this

In [ ]:
# ── Tests for Problem 8 ──────────────────────────────────────────
assert result_8 is not None, "result_8 is None — did you assign your DataFrame?"
assert hasattr(result_8, 'columns'), "result_8 must be a Spark DataFrame"
cols = [c.lower() for c in result_8.columns]
for col in ['host_id', 'total_revenue', 'revenue_rank', 'cumulative_revenue_pct', 'host_percentile']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_8.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_rank = result_8.agg(F.min('revenue_rank')).collect()[0][0]
assert min_rank == 1, f"Minimum revenue_rank should be 1, got {min_rank}"
max_cum = result_8.agg(F.max('cumulative_revenue_pct')).collect()[0][0]
assert abs(float(max_cum) - 100.0) < 0.5, f"Max cumulative_revenue_pct should be ~100, got {max_cum}"
assert result_8.agg(F.min('total_revenue')).collect()[0][0] > 0, "total_revenue must be positive"
print(f"Problem 8 passed ✓  ({cnt} hosts)")